# 14 (Kaggle) — MoE dùng chung trên chuỗi 4 token: Top-k MoE và Soft MoE

Thay khối fusion của **M0** (MoE-D: mỗi nguồn một expert riêng, gate trộn theo nguồn) bằng một lớp MoE **dùng chung** chạy
trên **chuỗi 4 token** `[clin, derm, demo, monet]` (4 × d). Mọi thứ khác giữ như M0: Swin-B-384 · `d_model 256` ·
overlay `r5_nobce` · clin + derm + metadata · 3 head long-tail (τ 0 / 0.5 / 1) · head phụ · modality dropout · 384 px ·
20 epoch, patience 5 · batch 8 × 8 = 64. **2 run, fold 0**; mốc **M0 fold 0**: val Dice **0.601**, AUC 0.933.

| Run | Khối fusion | Chi tiết |
|---|---|---|
| `swin_base_384__abl_TK__f0` | **Top-k MoE** (hướng F) | 4 expert MLP dùng chung; router chấm điểm **từng token**, chọn **top-2** expert, trộn theo xác suất router đã chuẩn hoá; load-balancing loss (α = 0.01) |
| `swin_base_384__abl_SM__f0` | **Soft MoE** (hướng G) | 4 expert × 1 slot; mỗi slot là **trộn mềm của cả 4 token** (softmax theo token), mỗi token đầu ra là **trộn mềm của các slot** (softmax theo slot); không route cứng, không cần loss cân bằng |

```
clin ─ Swin-B ─┐
derm ─ Swin-B ─┤  (dùng chung)
demo ─ MLP ────┼─ 4 token [4 × d] ─ modality dropout + embedding nguồn
MONET ─ MLP ───┘          │
              x + MoE(LN(x))   ← TK: top-2 trong 4 expert / token   |   SM: Soft MoE 4 slot
                          │
              trung bình 4 token ─ LN ─ z ─ 3 head long-tail ─ trung bình xác suất ─ top-1
```
**Khác M0:** ở M0 expert gắn cố định với nguồn và gate cho 1 trọng số / nguồn; ở đây expert không biết token đến từ nguồn
nào (chỉ qua embedding nguồn) và router tự phân công. Cột `expert_<nguồn>` trong `oof.csv` = expert chính của từng token
(TK: top-1 của router; SM: expert có trọng số combine lớn nhất).

**Thời gian (T4 x2, ước lượng):** khối fusion chỉ ~1M tham số, chi phí như M0 → ~2 giờ/run, 2 run song song trên 2 GPU
→ **~2–2.5 giờ**. **Cách chạy:** Settings → **GPU T4 x2**, **Internet: On** → **Save Version → Save & Run All (Commit)**;
tải zip ở tab *Output*.

In [ ]:
# ===================== CẤU HÌNH =====================
REPO_URL = "https://github.com/trong5nhan6/fussion.git"
BRANCH = "main"
DRIVE_URL = "https://drive.google.com/file/d/1lM1nB_4pq3Vg-GwlaTMQjtus_bpbMhGX/view?usp=drive_link"  # zip dữ liệu (tải công khai bằng gdown)
DATA_SOURCE = None    # hoặc đường dẫn Kaggle Dataset, vd "/kaggle/input/milk10k" (khi đó không tải từ link)

# Chạy tiếp: gắn output của version trước làm Input rồi liệt kê thư mục outputs của nó
RESTORE = []
TIME_BUDGET_H = 11.0

# ===================== CHỌN THÍ NGHIỆM =====================
EXPERIMENT = "experiments/stage11_token_moe.yaml"
BACKBONES = ["swin_base_384"]
VARIANTS = ["TK", "SM"]             # TK = top-k MoE · SM = Soft MoE
FOLDS = [0]                         # chạy đủ 5 fold cho cấu hình thắng: [0, 1, 2, 3, 4]

In [ ]:
# ===================== THAM SỐ TRAIN (recipe M0) =====================
# Ưu tiên cao nhất: ghi đè mọi config. T4 không có bf16 -> fp16 (như run M0 trên Kaggle, test 0.512).
PARAMS = {
    "train.batch_size": 8,              # T4 16GB; CUDA out of memory -> 4 và accum_steps 16
    "train.accum_steps": 8,             # batch hiệu dụng 64
    "model.grad_checkpointing": True,
    "train.amp_dtype": "fp16",
    "train.epochs": 20,
    "train.patience": 5,
    "moe.n_experts": 4,                 # số expert dùng chung
    "moe.top_k": 2,                     # TK: mỗi token qua top-k expert
    "moe.slots_per_expert": 1,          # SM: số slot mỗi expert (4 expert x 1 = 4 slot = 4 token)
    "seed": 42,
    "predict.tta": True,
}
RUN_SUFFIX = ""

## 1. Cài đặt

In [ ]:
import os, glob
from IPython.display import Markdown, display

REPO_DIR = "/kaggle/working/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
DATA_DIR = "/tmp/MILK10k"               # ngoài /kaggle/working -> dữ liệu không bị lưu vào output notebook
os.environ["MILK10K_ROOT"] = DATA_DIR   # code tự đọc biến này thay cho data.root trong config

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q gdown
!git log -1 --format="Commit: %h %s"

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| số GPU:", torch.cuda.device_count())
assert torch.cuda.device_count() >= 1, "Chưa bật GPU: Settings -> Accelerator -> GPU T4 x2"

## 2. Dữ liệu

In [ ]:
src = f"--source {DATA_SOURCE}" if DATA_SOURCE else f'--drive-url "{DRIVE_URL}"'
!python scripts/setup_data.py {src} --dest {DATA_DIR}

In [ ]:
# Kiểm tra nhanh top-k MoE / Soft MoE + lưới 2 run (~1 phút, không tải trọng số)
!python -m pytest tests/test_smoke.py -q -p no:cacheprovider -k "token_moe or stage11 or moe"

## 3. Danh sách run và cấu hình thực tế

In [ ]:
restore_arg = ("--restore " + " ".join(RESTORE)) if RESTORE else ""
for r in RESTORE:
    print(r, "->", "OK" if os.path.isdir(r) else "KHÔNG TỒN TẠI (kiểm tra lại đường dẫn Input)")

In [ ]:
import shlex, yaml

def to_set_args(params: dict) -> str:
    # Chuyển PARAMS thành chuỗi --set key=value cho scripts (bỏ qua giá trị None)
    items = []
    for k, v in params.items():
        if v is None:
            continue
        val = yaml.safe_dump(v, default_flow_style=True).strip().removesuffix("...").strip()
        items.append(shlex.quote(f"{k}={val}"))
    return ("--set " + " ".join(items)) if items else ""

set_arg = to_set_args(PARAMS)
suffix_arg = f"--suffix {RUN_SUFFIX}" if RUN_SUFFIX else ""
print("Override:", set_arg or "(không có)")
print("Hậu tố tên run:", RUN_SUFFIX or "(không có)")

In [ ]:
select_arg = (f"--backbones {','.join(BACKBONES)} --variants {','.join(VARIANTS)}"
              + (f" --folds {','.join(map(str, FOLDS))}" if FOLDS else ""))
n_runs = len(BACKBONES) * len(VARIANTS) * len(FOLDS)
print(f"{n_runs} run, ước tính ~{120 * n_runs / 2 / 60:.1f} giờ trên 2 GPU (giới hạn phiên: {TIME_BUDGET_H} giờ)")
!python scripts/run_grid.py {EXPERIMENT} --env configs/env/kaggle_t4.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --dry-run

## 4. Train

In [ ]:
!python scripts/run_grid.py {EXPERIMENT} --env configs/env/kaggle_t4.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --gpus auto --est-run-h 2 --time-budget-h {TIME_BUDGET_H}

## 5. Kết quả: so với M0 fold 0 + cách router phân công expert
Val fold 0 của từng run. Mốc M0 fold 0: Dice **0.601**, AUC 0.933. Bảng phân công: tỉ lệ lesion mà token của mỗi nguồn
được giao cho từng expert (expert chính) — nếu một expert ôm mọi token thì MoE đã sụp về 1 expert.
File nộp test của từng run: `outputs/<run>/submission.csv` (**chưa nộp** — chọn theo val trước).

In [ ]:
import json, glob
import pandas as pd
CLASSES = ["AKIEC", "BCC", "BEN_OTH", "BKL", "DF", "INF", "MAL_OTH", "MEL", "NV", "SCCKA", "VASC"]
M0_F0 = {"Dice val": 0.601, "AUC": 0.933}
rows, per = [], {}
for d in sorted(glob.glob("outputs/*__abl_TK__f*") + glob.glob("outputs/*__abl_SM__f*")):
    try:
        m = json.load(open(f"{d}/metrics.json")); oof = pd.read_csv(f"{d}/oof.csv")
    except FileNotFoundError:
        continue
    name, o = d.rstrip("/").split("/")[-1], m["overall"]
    rows.append({"run": name, "Dice val": o["dice"], "AUC": o["auc"], "Δ Dice vs M0 f0": o["dice"] - M0_F0["Dice val"]})
    y, pred = oof["label"].to_numpy(), oof[CLASSES].to_numpy().argmax(1)
    per[name] = {c: 2 * ((y == k) & (pred == k)).sum() / max((y == k).sum() + (pred == k).sum(), 1)
                 for k, c in enumerate(CLASSES)}
    ex = [c for c in oof.columns if c.startswith("expert_")]
    if ex:
        share = pd.DataFrame({c.removeprefix("expert_"): oof[c].value_counts(normalize=True) for c in ex}).fillna(0)
        share.index = [f"expert {int(i)}" for i in share.index]
        print(f"{name}: tỉ lệ expert chính theo nguồn"); display(share.T.round(2))
if rows:
    tab = pd.DataFrame(rows).set_index("run")
    display(tab.round(3)); tab.round(4).to_csv("outputs/stage11_summary.csv")
    print("Dice val theo lớp (fold 0):"); display(pd.DataFrame(per).T.round(3))
else:
    print("Chưa có run nào xong trong outputs/")

## Nén kết quả để tải về

In [ ]:
import zipfile, datetime
from pathlib import Path

zip_path = Path("/kaggle/working") / f"14_token_moe_kaggle_outputs_{datetime.datetime.now():%Y%m%d_%H%M}.zip"
src = Path(REPO_DIR) / "outputs"
n = 0
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(src.rglob("*")):
        if f.is_file() and f.suffix != ".pt":          # không đóng gói checkpoint
            z.write(f, Path("outputs") / f.relative_to(src))
            n += 1
print(f"Đã nén {n} file -> {zip_path} ({zip_path.stat().st_size / 2**20:.1f} MB)")
print("Tải về: tab Output của version notebook (bên phải) -> {} -> Download".format(zip_path.name))